# Cluster of Computers

This notebook shows how to create an isolated local Ethernet and connect compute nodes to it and use FABlib's user defined network configuration functionality. We will also examine how nodes from different sites can behave. 

*This notebook has been modified from the original FABRIC example on Create a Local Ethernet (Layer 2) Network: User Defined Configuration*


## Import the FABlib Library


In [ ]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config();

User defined configuration requires specify a subnet for the network and specifying the IP to use for each interface before the request is submitted. You can signal FABlib to confgure the user definded IPs by setting the interface's mode to `config` using the `iface1.set_mode('config')` function before submitting the request. With user defince configuration, FABlib will use the IP defined by the used and configure the device during the post boot configuration stage.  Optionally, you can add routes to the node before submitting the request.

## Create the Experiment Slice

The following creates two nodes with basic NICs connected to an isolated local Ethernet.  

Two nodes are created and one NIC component is added to each node.  This example uses components of model `NIC_Basic` which are SR-IOV Virtual Function on a 100 Gpbs Mellanox ConnectX-6 PCI device. The VF is accessed by the node via PCI passthrough. Other NIC models are listed below. When using dedicated PCI devices the whole physical device is allocated to one node and the device is accessed by the node using PCI passthrough. Calling the `get_interfaces()` method on a component will return a list of interfaces. Many dedicated NIC components may have more than one port.  Either port can be connected to the network.

User defined configuration requires specifying a subnet for the network and specifying the IP to use for each interface before the request is submitted. You can signal FABlib to configure the user defined IPs by setting the interface's mode to `config` using the `iface1.set_mode('config')` function before submitting the request. With user defined configuration, FABlib will use the IP defined by the user and configure the device during the post boot configuration stage.  Optionally, you can add routes to the node before submitting the request.


NIC component models options:
- NIC_Basic: 100 Gbps Mellanox ConnectX-6 SR-IOV VF (1 Port)
- NIC_ConnectX_5: 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device (2 Ports) 
- NIC_ConnectX_6: 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device (2 Ports) 

In [ ]:
slice_name = 'MyCluster'
site1 = fablib.get_random_site()
site2 = fablib.get_random_site()

print(f"Sites: {site1} {site2}")

node1_name = 'Node1'
node2_name = 'Node2'

network_name='net1'

In [ ]:
#Create Slice
slice = fablib.new_slice(name=slice_name)

# Network
net1 = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

# Node1
node1 = slice.add_node(name=node1_name, site=site1)
iface1 = node1.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
iface1.set_mode('config')
net1.add_interface(iface1)
iface1.set_ip_addr(IPv4Address("192.168.1.1"))

# Node2
node2 = slice.add_node(name=node2_name, site=site2)
iface2 = node2.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
iface2.set_mode('config')
net1.add_interface(iface2)
iface2.set_ip_addr(IPv4Address("192.168.1.2"))


#Submit Slice Request
slice.submit();

## Run the Experiment

With user defined configuration the slice is ready for experimentation after it becomes active.  Note that user defined configuration works well when saving slices to a file and reinstantiating the slice.  Configuration tasks can be stored in the saved slice, reducing the complexity of notebooks and other runtime steps.

We will find the ping round trip time for this pair of sites.  Your experiment should be more interesting!


In [ ]:
slice.update()

print("Slice state:", slice.get_state())
print("Slice stable:", slice.isStable())

for node in slice.get_nodes():
    print("----", node.get_name(), "----")
    print("reservation state:", node.get_reservation_state())
    print("management ip:", node.get_management_ip())
    print("username:", node.get_username())
    print("error:", node.get_error_message())
    print(node.get_ssh_command())

In [ ]:
slice = fablib.get_slice(slice_name)

node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           

node2_addr = node2.get_interface(network_name=network_name).get_ip_addr()

stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [ ]:
slice.delete()

## Building A Big Cluster

In [ ]:

# Get the resources helper
resources = fablib.get_resources()
resources.update()

# if you have multiple nodes and want adequate resources, you need find a suitable site

nodesReq = 4
coresReq = 8
ramReq = 16

totalCoreAvail = nodesReq * coresReq
totalRamAvail = nodesReq * ramReq

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail * 1.5  and ram >= totalRamAvail * 1.5:
        usableSite.append(site)

print(usableSite)

In [ ]:
slice = fablib.new_slice(name="BigCluster")
import random

for i in range(1, nodesReq + 1):
    siteName = random.choice(usableSite)
    print(f"getting node from {siteName}")
    node = slice.add_node(name=f"node{i}", site=siteName, cores=8, ram=8, disk=50)
slice.submit()          # blocking by default

In [ ]:
import time

while True:
    time.sleep(10)
    slice.update()
    slice_state = slice.get_state()
    print(f"Slice state: {slice_state}")
    if slice_state == "Closing":
        print(f"Need to find new site")
        break
    else: 
        print("Slice stable:", slice.isStable())
    nodes = slice.get_nodes()
    if all(node.get_management_ip() is not None for node in nodes):
        for node in nodes:
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

In [ ]:
for i in range(1,nodesReq):
    src = slice.get_node(name="node" + str(i))
    for j in range(i + 1,nodesReq + 1):
        des = slice.get_node(name="node" + str(j))           
        print(f"{src.get_name()} is pinging {des.get_name()} ========")
        des_addr = des.get_management_ip()
        stdout, stderr = src.execute(f'ping -c 2 {des_addr}')    

In [ ]:
slice.delete()

## Cluster with VLAN

In [ ]:
# Get the resources helper
resources = fablib.get_resources()
resources.update()

nodesReq = 4
coresReq = 8
ramReq = 16

totalCoreAvail = nodesReq * coresReq
totalRamAvail = nodesReq * ramReq

usableSite = []
siteList = resources.get_site_names()
for site in siteList:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores >= totalCoreAvail * 1.5  and ram >= totalRamAvail * 1.5:
        usableSite.append(site)

print(usableSite)

In [ ]:
import random

slice = fablib.new_slice(name="LANCluster")
siteName = random.choice(usableSite)
print(f"Site: {siteName}")
# Network

net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))

for i in range(1, nodesReq + 1):
    print(f"getting node from {siteName}")
    node = slice.add_node(name=f"node{i}", site=siteName, cores=8, ram=8, disk=50)
    iface = node.add_component(model='NIC_Basic', name='nic').get_interfaces()[0]
    iface.set_mode('config')
    net.add_interface(iface)
    iface.set_ip_addr(IPv4Address(f"192.168.1.{i}"))

slice.submit()  

In [ ]:
import time

while True:
    time.sleep(10)
    slice.update()
    slice_state = slice.get_state()
    print(f"Slice state: {slice_state}")
    if slice_state == "Closing":
        print(f"Need to find new site")
        break
    else: 
        print("Slice stable:", slice.isStable())
    nodes = slice.get_nodes()
    if all(node.get_management_ip() is not None for node in nodes):
        for node in nodes:
            print("----", node.get_name(), "----")
            print("reservation state:", node.get_reservation_state())
            print("management ip:", node.get_management_ip())
            print("username:", node.get_username())
            print("error:", node.get_error_message())
            print(node.get_ssh_command())
        break

In [ ]:
for i in range(1,nodesReq):
    src = slice.get_node(name="node" + str(i))
    for j in range(i + 1,nodesReq + 1):
        des = slice.get_node(name="node" + str(j))           
        print(f"{src.get_name()} is pinging {des.get_name()} ========")
        des_addr = des.get_interface(name=f"{des.get_name()}-nic-p1").get_ip_addr()
        stdout, stderr = src.execute(f'ping -c 2 {des_addr}')  